In [9]:
from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
    InputGuardrailTripwireTriggered,
)

from pydantic import BaseModel
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [11]:
class Check_Cheat(BaseModel):
    detected: bool
    explanation: str


cheat_detection_agent = Agent(
    name="Cheat Detection",
    instructions="""You are a cheat detection agent. Analyze the input for potential cheating behavior.
    like asking for answers for fill in the blanks or multiple choice questions.""",
    model="gpt-4o-mini",
    output_type=Check_Cheat,
)


@input_guardrail
async def cheat_detection_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    detection_result = await Runner.run(cheat_detection_agent, input)

    return GuardrailFunctionOutput(
        tripwire_triggered=detection_result.final_output.detected,
        output_info=detection_result.final_output,
    )


study_helper = Agent(
    name="Study Helper",
    instructions="You are a helpful study assistant. Provide concise and accurate answers to the user's questions based on your knowledge.",
    model="gpt-4o-mini",
    input_guardrails=[cheat_detection_guardrail],
)

In [ ]:
try:
    response = await Runner.run(
        study_helper, "Give me the name of 5 best wingers of all time in football"
    )
    print("No cheating detected.")
    print(f"Response: {response.final_output}")

except InputGuardrailTripwireTriggered as e:
    print("Cheating detected!")
    print(f"Details: {e}")

No cheating detected.
Response: Here are five of the best wingers of all time in football:

1. **Garrincha** - Brazilian legend known for his incredible dribbling and agility.
2. **Cristiano Ronaldo** - Versatile forward with numerous awards and accolades, known for his goal-scoring ability.
3. **Lionel Messi** - Often considered one of the greatest players ever, with extraordinary dribbling and playmaking skills.
4. **George Best** - Northern Irish star famous for his flair and creativity on the wing.
5. **Neymar** - Brazilian forward known for his skill, speed, and ability to change games.

These players have significantly impacted the game and are celebrated for their contributions as wingers.
